In [16]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# **Датасет**
Датасет для этого задания я сгенерировал самостоятельно. Пусть у нас есть 5000 пользователей, каждому из них соответствует непрерывная метрика duration (время просмотра) и дискретная метрика active_days (количество дней недели, когда пользователь был в сети).
Сгенерируем данные использования за 12 недель. Логнормальное распределение для view_duration выбрано потому, что оно наиболее точно отражает реальное потребление контента. Большинство пользователей смотрят умеренно, но есть небольшая группа задротов/безработных, чье время просмотра в разы превышает среднее.

Пусть первые 8 недель были достаточно стабильными, а на неделях 9-11 произошёл заметный скачок активности. На 12-й неделе всё снова вернулось к обычному времени просмотра. Попробуем выявить эти изменения с помощью методов PSI и Adversarial validation.

In [24]:
np.random.seed(42)
n_users = 5000
all_weeks_data = []

for w in range(1, 13):
    # 1-8: base, 9-11: drift, 12: base
    is_drift = True if w in [9, 10, 11] else False

    if not is_drift:
        duration = np.random.lognormal(mean=2.5, sigma=0.6, size=n_users)
        active_days = np.random.binomial(n=7, p=0.4, size=n_users)
    else:
        duration = np.random.lognormal(mean=2.8, sigma=0.5, size=n_users)
        active_days = np.random.binomial(n=7, p=0.5, size=n_users)

    all_weeks_data.append(pd.DataFrame({
        'week': w,
        'view_duration': duration,
        'active_days': active_days
    }))

display(all_weeks_data[0].head())

,week,view_duration,active_days
0,1,16.412258,2
1,1,11.212637,2
2,1,17.968372,3
3,1,30.381015,2
4,1,10.585742,2


# **Результаты PSI и Adversarial validation**

Как мы видим, PSI и Adversarial validation точно выявили изменения в распределениях. В периоды стабильности (1-8 недели, 9-11 недели) PSI значительно меньше 0.1, а ROC-AUC около 0.5 (то есть модель не может найти различия между выборками). Между 8-9 и 11-12 неделями происходят скачки - PSI становится больше 0.2, а ROC-AUC поднимается до 0.7 (то есть модель уже неплохо различает данные - значит есть заметные отличия).

In [25]:
def calculate_psi(expected, actual, is_discrete=False):
    if is_discrete:
        all_cats = sorted(set(expected) | set(actual))
        e_counts = expected.value_counts(normalize=True).reindex(all_cats, fill_value=0.0001)
        a_counts = actual.value_counts(normalize=True).reindex(all_cats, fill_value=0.0001)
        e_perc, a_perc = e_counts.values, a_counts.values
    else:
        breakpoints = np.percentile(expected, np.arange(0, 110, 10))
        breakpoints[0], breakpoints[-1] = -np.inf, np.inf
        e_perc = np.histogram(expected, bins=breakpoints)[0] / len(expected)
        a_perc = np.histogram(actual, bins=breakpoints)[0] / len(actual)

    a_perc = np.where(a_perc == 0, 0.0001, a_perc)
    return np.sum((e_perc - a_perc) * np.log(e_perc / a_perc))

def get_adversarial_auc(df1, df2):
    X = pd.concat([df1, df2]).drop(columns=['week'])
    y = np.array([0]*len(df1) + [1]*len(df2))
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    model = RandomForestClassifier(n_estimators=50, max_depth=4, random_state=42)
    model.fit(X_train, y_train)
    return roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])


stats_list = []

for i in range(1, len(all_weeks_data)):
    prev, curr = all_weeks_data[i-1], all_weeks_data[i]

    psi_c = calculate_psi(prev['view_duration'], curr['view_duration'])
    psi_d = calculate_psi(prev['active_days'], curr['active_days'], is_discrete=True)
    auc = get_adversarial_auc(prev, curr)

    stats_list.append({
        'Comparison': f'W{i} vs W{i+1}',
        'PSI_duration': round(psi_c, 4),
        'PSI_active_days': round(psi_d, 4),
        'Adversarial_AUC': round(auc, 4),
        'Status': 'DRIFT!' if (psi_c > 0.1 or psi_d > 0.1 or auc > 0.6) else 'Stable'
    })

df_results = pd.DataFrame(stats_list)
display(df_results)

,Comparison,PSI_duration,PSI_active_days,Adversarial_AUC,Status
0,W1 vs W2,0.0011,0.0079,0.4932,Stable
1,W2 vs W3,0.0025,0.0035,0.4956,Stable
2,W3 vs W4,0.0048,0.0038,0.5006,Stable
3,W4 vs W5,0.0093,0.0042,0.5104,Stable
4,W5 vs W6,0.0043,0.0031,0.4925,Stable
5,W6 vs W7,0.0029,0.0020,0.4994,Stable
6,W7 vs W8,0.0043,0.0028,0.5096,Stable
7,W8 vs W9,0.3525,0.2742,0.7075,DRIFT!
8,W9 vs W10,0.0033,0.0022,0.4917,Stable
9,W10 vs W11,0.0026,0.0060,0.5206,Stable
